In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
# CELL 1: Clone đúng repository/branch vào Kaggle working
import os
from pathlib import Path

REPO_URL = 'https://github.com/DanielQH07/tranSTR_Casual.git'
REPO_NAME = 'tranSTR_Casual'
BRANCH = 'master'
WORKING_DIR = Path('/kaggle/working')
REPO_DIR = WORKING_DIR / REPO_NAME

def _has_repo_files(path):
    path = Path(path)
    return (path / 'DataLoader.py').exists() and (path / 'networks' / 'model.py').exists()

if not _has_repo_files(Path.cwd()):
    if not REPO_DIR.exists():
        rc = os.system(f'git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}')
        if rc != 0:
            raise RuntimeError('git clone failed; kiểm tra Kaggle Internet setting')
    target = REPO_DIR / 'causalvid'
    os.chdir(target if target.exists() else REPO_DIR)

if not _has_repo_files(Path.cwd()):
    raise FileNotFoundError(f'Repository files not found in {Path.cwd()}')
print(f'Working directory: {Path.cwd()}')


Cloning into '/kaggle/working/tranSTR_Casual'...


Working directory: /kaggle/working/tranSTR_Casual


In [4]:
# CELL 2: Settings, dependencies và Kaggle authentication
print('=== CELL 2: Settings & Authentication ===')
import importlib
import os
import subprocess
import sys

USE_EMA_WEIGHTS = True
REQUIRE_GPU = True
STRICT_RAW_VIDEO_COVERAGE = True

WEIGHT_DATASET = '/kaggle/input/datasets/danielq07/gdinofrcnn-ncod-hum-model/best_model_gdinofrcnn_ncod_hum_run1_generic_safe_lora_hn_ema_cos.ckpt'
CHECKPOINT_FILENAME = 'best_model_gdinofrcnn_ncod_hum_run1_generic_safe_lora_hn_ema_cos.ckpt'
RAW_VIDEO_DATASET = '/kaggle/input/datasets/danielq07/causal-vidqa-raw-video-full'

packages = ['kagglehub', 'transformers', 'sentencepiece', 'einops', 'tqdm']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', *packages])

def _read_kaggle_secret(name):
    value = os.environ.get(name, '').strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return (UserSecretsClient().get_secret(name) or '').strip()
    except Exception:
        return ''

kaggle_token = _read_kaggle_secret('KAGGLE_API_TOKEN')
if kaggle_token:
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    print('KAGGLE_API_TOKEN loaded from Kaggle Secrets/environment')
else:
    print('No Kaggle token supplied; public datasets will use the Kaggle runtime identity')

required_modules = [
    'torch', 'numpy', 'pandas', 'tqdm', 'IPython',
    'einops', 'transformers', 'kagglehub',
]
missing = []
for module_name in required_modules:
    try:
        importlib.import_module(module_name)
    except Exception as exc:
        missing.append(f'{module_name}: {exc}')
if missing:
    raise ImportError('Dependency preflight failed:\n' + '\n'.join(missing))

print('Dependency/auth preflight OK')


=== CELL 2: Settings & Authentication ===
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 89.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
preprocessing 0.1.13 requires nltk==3.2.4, but you have nltk 3.9.2 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.13.1 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
fastai 2.8.4 requires fastcore<1.9,>=1.8.0, but you have fastcore 1.11.3 which is incompatible.


No Kaggle token supplied; public datasets will use the Kaggle runtime identity
Dependency/auth preflight OK


In [5]:
# CELL 3: Tải weight, feature, annotation, split và toàn bộ raw MP4 bằng KaggleHub
print('=== CELL 3: Resolve Data & Checkpoint ===')
import os
import shutil
from collections import Counter
from pathlib import Path
import kagglehub

def _resolve_input(direct_paths, slug, env_name=None):
    # 1) Explicit environment path, 2) attached Kaggle Input, 3) KaggleHub fallback.
    override = os.environ.get(env_name, '').strip() if env_name else ''
    if override:
        override_path = Path(override)
        if override_path.exists():
            print(f'Using {env_name}: {override_path}')
            return override_path
        print(f'Ignoring missing {env_name}: {override_path}')

    for candidate in direct_paths:
        candidate = Path(candidate)
        if candidate.exists():
            print(f'Using attached Kaggle input: {candidate}')
            return candidate

    # dataset_download only accepts owner/dataset, never owner/dataset/subfolder.
    parts = [part for part in str(slug).strip('/').split('/') if part]
    if len(parts) != 2:
        raise ValueError(f'Kaggle dataset handle must be owner/dataset, got: {slug}')
    print(f'Attached input not found; downloading Kaggle dataset: {slug}')
    return Path(kagglehub.dataset_download(slug))

def _find_dir_with_ext(root, extension):
    counts = {}
    for path in Path(root).rglob('*' + extension):
        counts[path.parent] = counts.get(path.parent, 0) + 1
    return max(counts, key=counts.get) if counts else None

def _find_named(root, name):
    root = Path(root)
    if root.name.lower() == name.lower():
        return root
    return next(
        (path for path in root.rglob('*') if path.is_dir() and path.name.lower() == name.lower()),
        None,
    )

# Direct Kaggle Input paths supplied for this notebook.
DIRECT_DINOV3_PATH = Path('/kaggle/input/datasets/danielq07/dinov3-feat/features')
DIRECT_GDINO_PATH = Path('/kaggle/input/datasets/danielq07/causal-vidqa-gdinofasterrcnn-features-merged/gdinofasterrcnn_features')
DIRECT_SPLIT_PATH = Path('/kaggle/input/casual-vid-data-split/split')
DIRECT_RAW_VIDEO_PATH = Path('/kaggle/input/datasets/danielq07/causal-vidqa-raw-video-full')

# Annotation/weight may appear with or without the /datasets/owner prefix depending on how they were attached.
DIRECT_ANNOTATION_PATHS = [
    Path('/kaggle/input/datasets/lusnaw/text-annotation/QA'),
    Path('/kaggle/input/text-annotation/QA'),
]
DIRECT_WEIGHT_PATHS = [
    Path('/kaggle/input/datasets/danielq07/gdinofrcnn-ncod-hum-model'),
    Path('/kaggle/input/gdinofrcnn-ncod-hum-model'),
]

# Input feature/QA/split used directly by TranSTR.
dinov3_root = _resolve_input(
    [DIRECT_DINOV3_PATH], 'danielq07/dinov3-feat', 'DINOV3_DATASET_PATH'
)
gdino_root = _resolve_input(
    [DIRECT_GDINO_PATH],
    'danielq07/causal-vidqa-gdinofasterrcnn-features-merged',
    'GDINO_DATASET_PATH',
)
annotation_root = _resolve_input(
    DIRECT_ANNOTATION_PATHS, 'lusnaw/text-annotation', 'ANNO_DATASET_PATH'
)
split_root = _resolve_input(
    [DIRECT_SPLIT_PATH], 'danielq07/casual-vid-data-split', 'SPLIT_DATASET_PATH'
)

# Full MP4 dataset; videos are not split or merged.
raw_dataset_root = _resolve_input(
    [DIRECT_RAW_VIDEO_PATH], RAW_VIDEO_DATASET, 'RAW_VIDEO_DATASET_PATH'
)
raw_video_files = sorted(raw_dataset_root.rglob('*.mp4'))
if not raw_video_files:
    raise FileNotFoundError(f'No raw MP4 files under {raw_dataset_root}')
raw_stem_counts = Counter(path.stem for path in raw_video_files)
duplicate_raw_ids = sorted(key for key, count in raw_stem_counts.items() if count > 1)
if duplicate_raw_ids:
    raise RuntimeError(f'Duplicate raw video IDs: {duplicate_raw_ids[:20]}')
RAW_VIDEO_MAP = {path.stem: path for path in raw_video_files}
RAW_VIDEO_ROOT = raw_dataset_root

# Dataset DINOv3 có thể chứa nhiều thư mục con; tạo view phẳng vì DataLoader đọc <video_id>.pt trực tiếp.
all_dinov3 = list(dinov3_root.rglob('*.pt'))
if not all_dinov3:
    raise FileNotFoundError(f'No DINOv3 .pt files under {dinov3_root}')
source_name_counts = Counter(source.name for source in all_dinov3)
duplicate_feature_names = sorted(name for name, count in source_name_counts.items() if count > 1)
if duplicate_feature_names:
    raise RuntimeError(f'Duplicate DINOv3 feature names: {duplicate_feature_names[:20]}')

CLIP_MERGED_PATH = Path('/kaggle/working/dinov3_features_flat')
CLIP_MERGED_PATH.mkdir(parents=True, exist_ok=True)
for source in all_dinov3:
    destination = CLIP_MERGED_PATH / source.name
    if destination.exists():
        continue
    try:
        destination.symlink_to(source)
    except Exception:
        shutil.copy2(source, destination)

merged_count = len(list(CLIP_MERGED_PATH.glob('*.pt')))
if merged_count != len(all_dinov3):
    raise RuntimeError(f'DINOv3 flat view incomplete: {merged_count}/{len(all_dinov3)}')

GDINO_MERGED_PATH = _find_dir_with_ext(gdino_root, '.pkl') or gdino_root
ANNOTATION_QA_PATH = _find_named(annotation_root, 'QA') or annotation_root
SPLIT_PATH = _find_named(split_root, 'split') or split_root

# Checkpoint cố định từ Kaggle dataset đã cung cấp.
weight_root = _resolve_input(
    DIRECT_WEIGHT_PATHS, WEIGHT_DATASET, 'WEIGHT_DATASET_PATH'
)
exact_weight_matches = list(weight_root.rglob(CHECKPOINT_FILENAME))
if len(exact_weight_matches) != 1:
    raise FileNotFoundError(
        f'Expected exactly one {CHECKPOINT_FILENAME} under {weight_root}, found {exact_weight_matches}'
    )
checkpoint_path = exact_weight_matches[0]

for name, path in (
    ('DINOv3', CLIP_MERGED_PATH),
    ('GDINO+FRCNN', GDINO_MERGED_PATH),
    ('QA', ANNOTATION_QA_PATH),
    ('split', SPLIT_PATH),
    ('raw MP4 root', RAW_VIDEO_ROOT),
    ('checkpoint', checkpoint_path),
):
    if not Path(path).exists():
        raise FileNotFoundError(f'{name} path not found: {path}')
    print(f'{name}: {path}')

print(f'DINOv3 files: {merged_count}')
print(f'Raw MP4 videos: {len(RAW_VIDEO_MAP)}')


=== CELL 3: Resolve Data & Checkpoint ===
Using attached Kaggle input: /kaggle/input/datasets/danielq07/dinov3-feat/features
Using attached Kaggle input: /kaggle/input/datasets/danielq07/causal-vidqa-gdinofasterrcnn-features-merged/gdinofasterrcnn_features
Using attached Kaggle input: /kaggle/input/text-annotation/QA
Using attached Kaggle input: /kaggle/input/casual-vid-data-split/split
Using attached Kaggle input: /kaggle/input/datasets/danielq07/causal-vidqa-raw-video-full
Using attached Kaggle input: /kaggle/input/datasets/danielq07/gdinofrcnn-ncod-hum-model
DINOv3: /kaggle/working/dinov3_features_flat
GDINO+FRCNN: /kaggle/input/datasets/danielq07/causal-vidqa-gdinofasterrcnn-features-merged/gdinofasterrcnn_features
QA: /kaggle/input/text-annotation/QA
split: /kaggle/input/casual-vid-data-split/split
raw MP4 root: /kaggle/input/datasets/danielq07/causal-vidqa-raw-video-full
checkpoint: /kaggle/input/datasets/danielq07/gdinofrcnn-ncod-hum-model/best_model_gdinofrcnn_ncod_hum_run1_gen

In [6]:
# CELL 4: Imports và inference helpers (Kaggle)
print('=== CELL 4: Imports & Helpers ===')
import json
import os
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader
from torch.utils.data._utils.collate import default_collate
from tqdm.auto import tqdm
from IPython.display import display

from utils.util import set_gpu_devices, set_seed
from DataLoader import VideoQADataset
from networks.model import VideoQAmodel

QUESTION_TYPES = [
    'counterfactual_reason', 'predictive_reason', 'counterfactual',
    'predictive', 'explanatory', 'descriptive'
]

def _unpack_batch(batch):
    if len(batch) == 7:
        return batch
    if len(batch) == 6:
        frame, obj, question, answers, answer_id, key = batch
        return frame, obj, question, answers, answer_id, key, None
    raise ValueError(f'Unexpected batch with {len(batch)} elements')

def _fit_object_sample(sample, expected_shape):
    items = list(sample)
    obj = torch.as_tensor(items[1]).float()
    if obj.ndim != 3:
        raise RuntimeError(f'Expected object feature [T,O,D], got {tuple(obj.shape)}')
    fixed = obj.new_zeros(expected_shape)
    sizes = tuple(min(obj.shape[index], expected_shape[index]) for index in range(3))
    fixed[:sizes[0], :sizes[1], :sizes[2]] = obj[:sizes[0], :sizes[1], :sizes[2]]
    items[1] = torch.nan_to_num(fixed, nan=0.0, posinf=0.0, neginf=0.0)
    return tuple(items)

def _safe_collate(batch):
    expected = (Config.input_frames, Config.objs, Config.obj_feat_dim)
    return default_collate([_fit_object_sample(sample, expected) for sample in batch])

def _split_qns_key(qns_key):
    key = str(qns_key)
    for question_type in QUESTION_TYPES:
        suffix = f'_{question_type}'
        if key.endswith(suffix):
            return key[:-len(suffix)], question_type
    return key, 'unknown'

def _compute_metrics(type_results):
    mapping = [
        ('Description', 'descriptive'),
        ('Explanation', 'explanatory'),
        ('Predictive-Answer', 'predictive'),
        ('Predictive-Reason', 'predictive_reason'),
        ('Counterfactual-Answer', 'counterfactual'),
        ('Counterfactual-Reason', 'counterfactual_reason'),
    ]
    metrics = {}
    for metric_name, question_type in mapping:
        rows = type_results.get(question_type, [])
        metrics[metric_name] = (
            100.0 * sum(1 for row in rows if row['correct']) / len(rows) if rows else 0.0
        )

    def paired(answer_type, reason_type):
        answers = {row['video_id']: row['correct'] for row in type_results.get(answer_type, [])}
        reasons = {row['video_id']: row['correct'] for row in type_results.get(reason_type, [])}
        common = set(answers) & set(reasons)
        if not common:
            return 0.0
        return 100.0 * sum(answers[vid] and reasons[vid] for vid in common) / len(common)

    metrics['PAR'] = paired('predictive', 'predictive_reason')
    metrics['CAR'] = paired('counterfactual', 'counterfactual_reason')
    metrics['Acc_ALL'] = (
        metrics['Description'] + metrics['Explanation'] + metrics['PAR'] + metrics['CAR']
    ) / 4.0
    return metrics

def _torch_load(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)

def _load_model_state_strict(model, state):
    if state and all(str(key).startswith('module.') for key in state):
        state = {str(key)[7:]: value for key, value in state.items()}
    current = model.state_dict()
    ignored = ('q_family_embed.', 'knowledge_head.', 'k_proj.')
    filtered = {}
    unexpected = []
    shape_errors = []
    for key, value in state.items():
        if key not in current:
            unexpected.append(key)
        elif current[key].shape != value.shape:
            shape_errors.append(
                f'{key}: checkpoint={tuple(value.shape)} model={tuple(current[key].shape)}'
            )
        else:
            filtered[key] = value
    missing = [key for key in current if key not in filtered]
    critical_missing = [key for key in missing if not key.startswith(ignored)]
    critical_unexpected = [key for key in unexpected if not key.startswith(ignored)]
    if shape_errors or critical_missing or critical_unexpected:
        raise RuntimeError(
            'Incompatible checkpoint:\n' + '\n'.join(shape_errors[:20])
            + f'\nmissing={critical_missing[:20]}\nunexpected={critical_unexpected[:20]}'
        )
    model.load_state_dict(filtered, strict=False)
    return missing, unexpected

print('Imports and helpers ready')


=== CELL 4: Imports & Helpers ===
Imports and helpers ready


In [7]:
# CELL 5: Cấu hình run1 cố định cho Kaggle
print('=== CELL 5: Fixed Run1 Config ===')

class Config:
    input_frames = 16
    objs = 12
    selected_frames = 5
    topk_objects = 5
    frame_feat_dim = 1024
    obj_feat_dim = 2820
    n_query = 5

    use_grounding_dino = True
    obj_use_bbox_pos_embed = True
    obj_bbox_dim = 4
    obj_hard_gather_from_frame = True
    obj_split_roi_class = False
    obj_roi_dim = 2048
    obj_class_dim = 768
    obj_mask_padding = True
    hard_eval = False

    d_model = 768
    word_dim = 768
    nheads = 8
    num_encoder_layers = 2
    normalize_before = True
    activation = 'gelu'
    dropout = 0.3
    encoder_dropout = 0.3
    text_encoder_type = 'microsoft/deberta-base'
    freeze_text_encoder = False
    num_question_families = 6
    lambda_knowledge = 0.0

    batch_size = 32
    num_workers = 0
    pin_memory = False
    seed = 999
    gpu = 0

RUN_TAG = 'run1_kaggle_bs32_ncod_hum_verifier'
OUTPUT_DIR = Path('/kaggle/working/inference_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_OUTPUT_PATH = OUTPUT_DIR / f'test_predictions_{RUN_TAG}.csv'
METRICS_OUTPUT_PATH = OUTPUT_DIR / f'inference_metrics_{RUN_TAG}.json'

set_gpu_devices(Config.gpu)
set_seed(Config.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if REQUIRE_GPU and device.type != 'cuda':
    raise RuntimeError('GPU runtime is required. Select a GPU in Kaggle Notebook settings.')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CSV output: {CSV_OUTPUT_PATH}')


=== CELL 5: Fixed Run1 Config ===
Device: cuda
GPU: Tesla T4
CSV output: /kaggle/working/inference_output/test_predictions_run1_kaggle_bs32_ncod_hum_verifier.csv


In [8]:
# CELL 6: Tạo test dataset, kiểm tra raw MP4 coverage và DataLoader
print('=== CELL 6: Test Dataset & Raw Video Coverage ===')

test_dataset = VideoQADataset(
    split='test',
    n_query=Config.n_query,
    obj_num=Config.objs,
    sample_list_path=str(ANNOTATION_QA_PATH),
    video_feature_path=str(CLIP_MERGED_PATH),
    grounding_dino_path=str(GDINO_MERGED_PATH),
    split_dir=str(SPLIT_PATH),
    topK_frame=Config.input_frames,
    max_samples=None,
    verbose=True,
    return_family_id=True,
)
if len(test_dataset) == 0:
    raise RuntimeError('Test dataset is empty; verify CELL 3 paths.')

TEST_VIDEO_IDS = sorted({str(value) for value in test_dataset.sample_list['video_id'].tolist()})
missing_raw_videos = [video_id for video_id in TEST_VIDEO_IDS if video_id not in RAW_VIDEO_MAP]
print(f'Test videos: {len(TEST_VIDEO_IDS)} | raw MP4 matched: {len(TEST_VIDEO_IDS) - len(missing_raw_videos)}')
if missing_raw_videos:
    message = f'Missing raw MP4 for {len(missing_raw_videos)} test videos: {missing_raw_videos[:20]}'
    if STRICT_RAW_VIDEO_COVERAGE:
        raise FileNotFoundError(message)
    print('WARNING:', message)

raw_probe = test_dataset[0]
fixed_probe = _fit_object_sample(
    raw_probe, (Config.input_frames, Config.objs, Config.obj_feat_dim)
)
print('Object feature guard:', tuple(raw_probe[1].shape), '->', tuple(fixed_probe[1].shape))
assert tuple(fixed_probe[1].shape) == (16, 12, 2820)

test_loader = DataLoader(
    test_dataset,
    batch_size=Config.batch_size,
    shuffle=False,
    num_workers=Config.num_workers,
    pin_memory=Config.pin_memory,
    collate_fn=_safe_collate,
)
print(f'Test samples: {len(test_dataset)} | batches: {len(test_loader)}')


=== CELL 6: Test Dataset & Raw Video Coverage ===
[test] Object feature format: unknown
[test] Using GroundingDINO features
[test] Indexed 26900 object features, 26900 ViT features
[test] Loaded 5429 video IDs from /kaggle/input/casual-vid-data-split/split/test.pkl
[test] ViT: 5429, Obj: 5429, Both: 5429


[test] Parsing annotations:   0%|          | 0/5429 [00:00<?, ?it/s]

[test] Final: 32574 QA pairs
Test videos: 5429 | raw MP4 matched: 5429
Object feature guard: (16, 12, 2820) -> (16, 12, 2820)
Test samples: 32574 | batches: 1018


In [9]:
# CELL 7: Dựng model và load Kaggle checkpoint
print('=== CELL 7: Load Checkpoint & Model ===')

if not checkpoint_path.is_file():
    raise FileNotFoundError(f'Kaggle checkpoint not found: {checkpoint_path}')

model_kwargs = {
    'text_encoder_type': Config.text_encoder_type,
    'freeze_text_encoder': Config.freeze_text_encoder,
    'n_query': Config.n_query,
    'objs': Config.objs,
    'frames': Config.input_frames,
    'topK_frame': Config.selected_frames,
    'topK_obj': Config.topk_objects,
    'hard_eval': Config.hard_eval,
    'frame_feat_dim': Config.frame_feat_dim,
    'obj_feat_dim': Config.obj_feat_dim,
    'use_grounding_dino': Config.use_grounding_dino,
    'obj_use_bbox_pos_embed': Config.obj_use_bbox_pos_embed,
    'obj_hard_gather_from_frame': Config.obj_hard_gather_from_frame,
    'obj_bbox_dim': Config.obj_bbox_dim,
    'obj_split_roi_class': Config.obj_split_roi_class,
    'obj_roi_dim': Config.obj_roi_dim,
    'obj_class_dim': Config.obj_class_dim,
    'obj_mask_padding': Config.obj_mask_padding,
    'd_model': Config.d_model,
    'word_dim': Config.word_dim,
    'nheads': Config.nheads,
    'num_encoder_layers': Config.num_encoder_layers,
    'normalize_before': Config.normalize_before,
    'activation': Config.activation,
    'dropout': Config.dropout,
    'encoder_dropout': Config.encoder_dropout,
    'num_question_families': Config.num_question_families,
    'lambda_knowledge': Config.lambda_knowledge,
    'device': device,
}
model = VideoQAmodel(**model_kwargs)
for module_name in ('q_family_embed', 'knowledge_head', 'k_proj'):
    if hasattr(model, module_name):
        delattr(model, module_name)
model.to(device)

checkpoint = _torch_load(checkpoint_path, device)
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    if USE_EMA_WEIGHTS and checkpoint.get('ema_model_state_dict') is not None:
        selected_state = checkpoint['ema_model_state_dict']
        WEIGHT_STATE = 'ema_model_state_dict'
    else:
        selected_state = checkpoint['model_state_dict']
        WEIGHT_STATE = 'model_state_dict'
    BEST_VAL_ACC = float(checkpoint.get('best_acc', 0.0))
    BEST_EPOCH = int(checkpoint.get('best_epoch', checkpoint.get('epoch', 0)))
else:
    selected_state = checkpoint
    WEIGHT_STATE = 'raw_state_dict'
    BEST_VAL_ACC = 0.0
    BEST_EPOCH = 0

missing, unexpected = _load_model_state_strict(model, selected_state)
model.eval()
print(f'Checkpoint: {checkpoint_path}')
print(f'Weight state: {WEIGHT_STATE}')
print(f'Best epoch: {BEST_EPOCH} | best val Acc_ALL: {BEST_VAL_ACC:.2f}%')
print(f'Ignored legacy knowledge keys: missing={len(missing)}, unexpected={len(unexpected)}')


=== CELL 7: Load Checkpoint & Model ===


config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  559MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

...

model.safetensors: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

Checkpoint: /kaggle/input/datasets/danielq07/gdinofrcnn-ncod-hum-model/best_model_gdinofrcnn_ncod_hum_run1_generic_safe_lora_hn_ema_cos.ckpt
Weight state: ema_model_state_dict
Best epoch: 6 | best val Acc_ALL: 64.82%
Ignored legacy knowledge keys: missing=0, unexpected=7


In [10]:
# CELL 8: Inference toàn bộ test split/video và xuất CSV đầy đủ
print('=== CELL 8: Full Test Inference ===')

metadata = {}
for _, row in test_dataset.sample_list.iterrows():
    video_id = str(row.get('video_id', ''))
    question_type = str(row.get('type', 'unknown'))
    key = f'{video_id}_{question_type}'
    metadata[key] = {
        'video_id': video_id,
        'question_type': question_type,
        'question': str(row.get('question', '')),
        'answers': [str(row.get(f'a{index}', '')) for index in range(Config.n_query)],
        'raw_video_path': str(RAW_VIDEO_MAP.get(video_id, '')),
    }

prediction_rows = []
type_results = {}
with torch.inference_mode():
    for batch in tqdm(test_loader, desc='Inference'):
        frame, obj, questions, answer_words, answer_ids, keys, _ = _unpack_batch(batch)
        output = model(
            frame.to(device),
            obj.to(device),
            questions,
            answer_words,
            return_aux=True,
            q_family_id=None,
        )
        probabilities = torch.softmax(output['logits'], dim=-1).cpu().numpy()
        predictions = probabilities.argmax(axis=-1)
        targets = answer_ids.numpy()

        for key, prediction, target, probability in zip(
            keys, predictions, targets, probabilities
        ):
            key = str(key)
            fallback_video, fallback_type = _split_qns_key(key)
            meta = metadata.get(key, {})
            video_id = str(meta.get('video_id', fallback_video))
            question_type = str(meta.get('question_type', fallback_type))
            candidates = list(meta.get('answers', [''] * Config.n_query))[:Config.n_query]
            candidates += [''] * (Config.n_query - len(candidates))
            prediction = int(prediction)
            target = int(target)
            correct = prediction == target
            type_results.setdefault(question_type, []).append(
                {'video_id': video_id, 'correct': correct}
            )
            result = {
                'run_tag': RUN_TAG,
                'checkpoint_file': checkpoint_path.name,
                'weight_state': WEIGHT_STATE,
                'best_epoch': BEST_EPOCH,
                'best_val_acc': BEST_VAL_ACC,
                'qns_key': key,
                'video_id': video_id,
                'question_type': question_type,
                'question': str(meta.get('question', '')),
                'raw_video_path': str(meta.get('raw_video_path', RAW_VIDEO_MAP.get(video_id, ''))),
                'raw_video_exists': int(video_id in RAW_VIDEO_MAP),
                'correct_idx': target,
                'correct_answer': candidates[target],
                'predicted_idx': prediction,
                'predicted_answer': candidates[prediction],
                'is_correct': int(correct),
                'confidence': float(probability[prediction]),
            }
            for index in range(Config.n_query):
                result[f'a{index}'] = candidates[index]
                result[f'prob_a{index}'] = float(probability[index])
            prediction_rows.append(result)

if len(prediction_rows) != len(test_dataset):
    raise RuntimeError(
        f'Incomplete inference: rows={len(prediction_rows)}, test_samples={len(test_dataset)}'
    )

base_columns = [
    'run_tag', 'checkpoint_file', 'weight_state', 'best_epoch', 'best_val_acc',
    'qns_key', 'video_id', 'question_type', 'question', 'raw_video_path', 'raw_video_exists',
]
answer_columns = [f'a{index}' for index in range(Config.n_query)]
result_columns = [
    'correct_idx', 'correct_answer', 'predicted_idx', 'predicted_answer',
    'is_correct', 'confidence',
]
probability_columns = [f'prob_a{index}' for index in range(Config.n_query)]
prediction_df = pd.DataFrame(prediction_rows)[
    base_columns + answer_columns + result_columns + probability_columns
]
prediction_df.to_csv(CSV_OUTPUT_PATH, index=False, encoding='utf-8')

metrics = _compute_metrics(type_results)
metrics['Plain_Acc'] = 100.0 * float(prediction_df['is_correct'].mean())
metrics['WeightedScore_WeakPriority'] = (
    0.35 * metrics['Predictive-Reason']
    + 0.35 * metrics['Counterfactual-Reason']
    + 0.20 * metrics['Acc_ALL']
    + 0.10 * BEST_VAL_ACC
)
with METRICS_OUTPUT_PATH.open('w', encoding='utf-8') as handle:
    json.dump(metrics, handle, indent=2, ensure_ascii=False)

print(f'Rows: {len(prediction_df)} / {len(test_dataset)}')
print(f"PAR={metrics['PAR']:.2f}% | CAR={metrics['CAR']:.2f}% | Acc_ALL={metrics['Acc_ALL']:.2f}%")
print(f'CSV: {CSV_OUTPUT_PATH}')
print(f'Metrics: {METRICS_OUTPUT_PATH}')
display(prediction_df.head())


=== CELL 8: Full Test Inference ===


Inference:   0%|          | 0/1018 [00:00<?, ?it/s]

Rows: 32574 / 32574
PAR=53.14% | CAR=52.33% | Acc_ALL=64.63%
CSV: /kaggle/working/inference_output/test_predictions_run1_kaggle_bs32_ncod_hum_verifier.csv
Metrics: /kaggle/working/inference_output/inference_metrics_run1_kaggle_bs32_ncod_hum_verifier.json


,run_tag,checkpoint_file,weight_state,best_epoch,best_val_acc,qns_key,video_id,question_type,question,raw_video_path,...,correct_answer,predicted_idx,predicted_answer,is_correct,confidence,prob_a0,prob_a1,prob_a2,prob_a3,prob_a4
0,run1_kaggle_bs32_ncod_hum_verifier,best_model_gdinofrcnn_ncod_hum_run1_generic_sa...,ema_model_state_dict,6,64.823748,--GF746y6UM_000496_000506_descriptive,--GF746y6UM_000496_000506,descriptive,What is [person_1] doing?,/kaggle/input/datasets/danielq07/causal-vidqa-...,...,[person_1] is playing the bass guitar on his h...,4,[person_1] is playing the bass guitar on his h...,1,0.981274,0.000624,0.005904,0.011960,0.000238,0.981274
1,run1_kaggle_bs32_ncod_hum_verifier,best_model_gdinofrcnn_ncod_hum_run1_generic_sa...,ema_model_state_dict,6,64.823748,--GF746y6UM_000496_000506_explanatory,--GF746y6UM_000496_000506,explanatory,Why is [person_1] wearing the headphones?,/kaggle/input/datasets/danielq07/causal-vidqa-...,...,[person_1] is listening to the music beat thro...,4,[person_1] is listening to the music beat thro...,1,0.996510,0.001126,0.000793,0.000797,0.000774,0.996510
2,run1_kaggle_bs32_ncod_hum_verifier,best_model_gdinofrcnn_ncod_hum_run1_generic_sa...,ema_model_state_dict,6,64.823748,--GF746y6UM_000496_000506_predictive,--GF746y6UM_000496_000506,predictive,What is [person_1] going to do?,/kaggle/input/datasets/danielq07/causal-vidqa-...,...,[person_1] will keep performing music with the...,2,[person_1] will keep performing music with the...,1,0.998190,0.001095,0.000138,0.998190,0.000161,0.000416
3,run1_kaggle_bs32_ncod_hum_verifier,best_model_gdinofrcnn_ncod_hum_run1_generic_sa...,ema_model_state_dict,6,64.823748,--GF746y6UM_000496_000506_predictive_reason,--GF746y6UM_000496_000506,predictive_reason,What is [person_1] going to do?,/kaggle/input/datasets/danielq07/causal-vidqa-...,...,[person_1] is plucking the bass guitar strings.,4,[person_1] is plucking the bass guitar strings.,1,0.996163,0.000990,0.001392,0.000659,0.000796,0.996163
4,run1_kaggle_bs32_ncod_hum_verifier,best_model_gdinofrcnn_ncod_hum_run1_generic_sa...,ema_model_state_dict,6,64.823748,--GF746y6UM_000496_000506_counterfactual,--GF746y6UM_000496_000506,counterfactual,What will happen if [person_1] does not have t...,/kaggle/input/datasets/danielq07/causal-vidqa-...,...,[person_1] will play bass guitar with a differ...,3,[person_1] will play bass guitar with a differ...,1,0.998024,0.000470,0.000576,0.000372,0.998024,0.000558


In [11]:
# CELL 9: Kiểm tra output Kaggle
print('=== CELL 9: Output Check ===')
if not CSV_OUTPUT_PATH.is_file() or CSV_OUTPUT_PATH.stat().st_size == 0:
    raise RuntimeError(f'CSV was not created correctly: {CSV_OUTPUT_PATH}')
if not METRICS_OUTPUT_PATH.is_file():
    raise RuntimeError(f'Metrics JSON not found: {METRICS_OUTPUT_PATH}')

print(f'Complete CSV: {CSV_OUTPUT_PATH} ({CSV_OUTPUT_PATH.stat().st_size / 1e6:.2f} MB)')
print(f'Metrics JSON: {METRICS_OUTPUT_PATH}')
print(f'Raw MP4 coverage: {len(TEST_VIDEO_IDS) - len(missing_raw_videos)}/{len(TEST_VIDEO_IDS)} test videos')
print('Chọn Save Version để giữ inference_output trong Kaggle Notebook Output.')


=== CELL 9: Output Check ===
Complete CSV: /kaggle/working/inference_output/test_predictions_run1_kaggle_bs32_ncod_hum_verifier.csv (26.12 MB)
Metrics JSON: /kaggle/working/inference_output/inference_metrics_run1_kaggle_bs32_ncod_hum_verifier.json
Raw MP4 coverage: 5429/5429 test videos
Chọn Save Version để giữ inference_output trong Kaggle Notebook Output.
